In [1]:
# ============================================================
# MODULE 2 - TITANIC MACHINE LEARNING
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

# IMPORTANT:
# Do NOT use sns.load_dataset() here.
# We use the CSV created by 01_eda.ipynb.

df = pd.read_csv("titanic.csv")

print("Dataset shape:", df.shape)
df.head()

Libraries imported successfully.
Dataset shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
# ============================================================
# CLASSIFICATION FEATURES AND TARGET
# ============================================================

classification_features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]

target = "survived"

X = df[classification_features].copy()
y = df[target].copy()

print("Features:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(
    (y.value_counts(normalize=True) * 100).round(2)
)

Features:
['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']

X shape: (891, 7)
y shape: (891,)

Target distribution:
survived
0    549
1    342
Name: count, dtype: int64

Target percentage:
survived
0    61.62
1    38.38
Name: proportion, dtype: float64


In [3]:
# ============================================================
# STRATIFIED TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(
    (y_train.value_counts(normalize=True) * 100).round(2)
)

print("\nTesting target distribution:")
print(
    (y_test.value_counts(normalize=True) * 100).round(2)
)

Training samples: 712
Testing samples: 179

Training target distribution:
survived
0    61.66
1    38.34
Name: proportion, dtype: float64

Testing target distribution:
survived
0    61.45
1    38.55
Name: proportion, dtype: float64


### Stratified Train/Test Split

The dataset was split into training and testing sets before fitting any preprocessing transformations. Stratification was performed using the survival target so that the class proportions remain similar in both datasets. This ordering helps prevent information from the test set from leaking into preprocessing fitted on the training data.

In [5]:
# ============================================================
# FEATURE GROUPS
# ============================================================

numeric_features = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "fare"
]

categorical_features = [
    "sex",
    "embarked"
]

print("Numeric Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numeric Features:
['pclass', 'age', 'sibsp', 'parch', 'fare']

Categorical Features:
['sex', 'embarked']


In [6]:
# ============================================================
# PREPROCESSING PIPELINES
# ============================================================

# Numeric preprocessing:
# 1. Impute missing values using training-data median
# 2. Standardize numeric features

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical preprocessing:
# 1. Impute missing categories using most frequent value
# 2. One-hot encode categories

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [7]:
# ============================================================
# LOGISTIC REGRESSION PIPELINE
# ============================================================

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

logistic_pipeline.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [8]:
# ============================================================
# DECISION TREE PIPELINE
# ============================================================

decision_tree_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=4,
                random_state=42
            )
        )
    ]
)

decision_tree_pipeline.fit(
    X_train,
    y_train
)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [12]:
# ============================================================
# CLASSIFIER EVALUATION FUNCTION
# ============================================================

def evaluate_classifier(
    model_name,
    model,
    X_test,
    y_test
):
    predictions = model.predict(X_test)

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc
    }

In [14]:
# ============================================================
# RANDOM FOREST PIPELINE
# ============================================================

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

random_forest_pipeline.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [15]:
print("Logistic Regression:", type(logistic_pipeline).__name__)
print("Decision Tree:", type(decision_tree_pipeline).__name__)
print("Random Forest:", type(random_forest_pipeline).__name__)

Logistic Regression: Pipeline
Decision Tree: Pipeline
Random Forest: Pipeline


In [16]:
# ============================================================
# EVALUATE ALL CLASSIFIERS
# ============================================================

classification_results = []

classification_results.append(
    evaluate_classifier(
        "Logistic Regression",
        logistic_pipeline,
        X_test,
        y_test
    )
)

classification_results.append(
    evaluate_classifier(
        "Decision Tree",
        decision_tree_pipeline,
        X_test,
        y_test
    )
)

classification_results.append(
    evaluate_classifier(
        "Random Forest",
        random_forest_pipeline,
        X_test,
        y_test
    )
)

classification_results_df = pd.DataFrame(
    classification_results
)

classification_results_df = (
    classification_results_df
    .set_index("Model")
    .round(4)
)

print("Classifier Comparison:")
display(classification_results_df)

Classifier Comparison:


,Accuracy,Precision,Recall,F1,ROC_AUC
Model,,,,,
Logistic Regression,0.8045,0.7931,0.6667,0.7244,0.8437
Decision Tree,0.7933,0.8636,0.5507,0.6726,0.8292
Random Forest,0.8156,0.8000,0.6957,0.7442,0.8300
